# MEIKA Vox · Santa María
**Escribe los nombres de tus audios en AUDIO_NAMES**, activa GPU y pulsa Entorno de ejecución → Ejecutar todo.** Autoriza Drive cuando Colab lo solicite.

Busca los dos audios por nombre dentro de Mi unidad y guarda el texto en `MEIKA_Vox/Transcripciones`.
La primera ejecución instala el motor y descarga modelos; requiere internet y puede tardar varios minutos.

Por defecto obtiene texto y marcas de tiempo **sin pedir tokens**. Para separar hablantes,
activa `DIARIZE` en la celda de configuración: requiere un HF_TOKEN y aceptar las condiciones
[del modelo pyannote](https://huggingface.co/pyannote/speaker-diarization-community-1).
Las etiquetas automáticas no identifican los nombres de las personas.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO = Path('/content/meika-vox')
RUNTIME = Path('/content/meika-vox-runtime')

def checked(cmd, **kwargs):
    subprocess.run([str(item) for item in cmd], check=True, **kwargs)

checked(['apt-get', '-qq', 'update'])
checked(['apt-get', '-qq', 'install', '-y', 'ffmpeg', 'git', 'python3-venv'])
if not (REPO / '.git').exists():
    checked(['git', 'clone', 'https://github.com/MeikaLab/meika-vox.git', REPO])
else:
    checked(['git', '-C', REPO, 'pull', '--ff-only'])
if not (RUNTIME / 'bin/python').exists():
    checked([sys.executable, '-m', 'venv', RUNTIME])
PYTHON = RUNTIME / 'bin/python'
VOX = RUNTIME / 'bin/meika-vox'
checked([PYTHON, '-m', 'pip', 'install', '--upgrade', 'pip'])
checked([PYTHON, '-m', 'pip', 'install', '-e', str(REPO) + '[whisperx]'])
checked([PYTHON, '-c', "import nltk; nltk.download('punkt_tab', raise_on_error=True)"])
checked([VOX, 'doctor', '--strict'])
print('Instalación comprobada. Ahora conecta Google Drive.')

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

## Configuración
Escribe los nombres de tus archivos en AUDIO_NAMES. No necesitas editar rutas si son únicos en Mi unidad.
Si encuentra más de una copia, indica la ruta completa en `AUDIO_NAMES`.
Sin GPU usa `small` en CPU; con GPU usa `large-v3`. La calidad de ambos modelos puede variar.

In [ ]:
import json

DIARIZE = False
AUDIO_NAMES = ['audio_1.m4a', 'audio_2.m4a']  # Escribe aquí los nombres de tus audios
ROOT = Path('/content/drive/MyDrive')
OUTPUT = ROOT / 'MEIKA_Vox/Transcripciones'
GLOSSARY = REPO / 'configs/glossaries/santa_maria_2026.json'

if DIARIZE:
    from getpass import getpass
    try:
        from google.colab import userdata
        token = userdata.get('HF_TOKEN')
    except Exception:
        token = None
    token = token or getpass('HF_TOKEN (oculto): ').strip()
    if not token:
        raise RuntimeError('Falta HF_TOKEN. Desactiva DIARIZE para obtener el texto sin hablantes.')
    os.environ['HF_TOKEN'] = token

report = json.loads(subprocess.check_output([str(VOX), 'doctor'], text=True))
if not report['asr_ready']:
    raise RuntimeError('El motor no está listo: vuelve a ejecutar la instalación.')
if DIARIZE and not report['diarization_ready']:
    raise RuntimeError('La separación de hablantes no está lista. Revisa HF_TOKEN.')
DEVICE = 'cuda' if report['cuda_available'] else 'cpu'
MODEL = 'large-v3' if DEVICE == 'cuda' else 'small'
BATCH_SIZE = 4 if DEVICE == 'cuda' else 1
print(f'Motor: {MODEL} · {DEVICE} · separar hablantes: {DIARIZE}')

AUDIOS = []
for name in AUDIO_NAMES:
    supplied = Path(name)
    if supplied.is_absolute():
        matches = [supplied] if supplied.is_file() else []
    else:
        matches = [p for p in ROOT.rglob(name) if p.is_file()]
    if len(matches) != 1:
        choices = '\n'.join(str(p) for p in matches)
        raise RuntimeError(f'{name}: encontré {len(matches)} archivos. '
                           f'Indica su ruta completa en AUDIO_NAMES.\n{choices}')
    AUDIOS.append(matches[0])
for audio in AUDIOS:
    print('Audio encontrado:', audio)
OUTPUT.mkdir(parents=True, exist_ok=True)

## Transcribir y guardar
Procesa ambos archivos completos. Un fallo en uno no impide intentar el otro.
Solo anuncia éxito si el proceso termina y produce una transcripción con texto.

In [ ]:
SUCCESSFUL_RUNS = []
FAILURES = []
for audio in AUDIOS:
    before = set(OUTPUT.rglob('manifest.json'))
    cmd = [VOX, 'transcribe', audio, '--project-id', 'SM26',
           '--language', 'es', '--model', MODEL, '--device', DEVICE,
           '--batch-size', str(BATCH_SIZE), '--glossary', GLOSSARY,
           '--output', OUTPUT]
    if DIARIZE:
        cmd.append('--diarize')
    print('Procesando:', audio.name, flush=True)
    try:
        checked(cmd)
        created = set(OUTPUT.rglob('manifest.json')) - before
        if len(created) != 1:
            raise RuntimeError('No apareció un único resultado nuevo.')
        run = created.pop().parent
        manifest = json.loads((run / 'manifest.json').read_text())
        transcript = run / 'transcript_normalized.txt'
        if manifest['counts']['segments'] == 0 or not transcript.read_text().strip():
            raise RuntimeError('El motor terminó pero no produjo texto. Revisa el audio.')
        SUCCESSFUL_RUNS.append((audio.name, run))
        print('Texto guardado:', transcript)
    except Exception as exc:
        FAILURES.append((audio.name, str(exc)))
        print('No se completó:', audio.name, str(exc))

## Resultados de esta ejecución
El TXT contiene el texto para leer; `qa.json` registra alertas que requieren revisión.
Sin diarización, los hablantes figuran como desconocidos. Revisa nombres y pasajes dudosos con el audio original.

In [ ]:
for name, run in SUCCESSFUL_RUNS:
    print('\nCOMPLETADO:', name)
    print('Carpeta:', run)
    for filename in ['transcript_normalized.txt', 'transcript_raw.jsonl',
                     'words.jsonl', 'speaker_turns.jsonl', 'qa.json', 'audio_qa.json']:
        print('  ', filename, '✅' if (run / filename).is_file() else 'FALTA')
    print('\nVista previa:\n', (run / 'transcript_normalized.txt').read_text()[:1500])
if FAILURES:
    print('\nNo se completaron:', FAILURES)
    raise RuntimeError('Hay audios pendientes; revisa los errores de la celda anterior.')
if not SUCCESSFUL_RUNS:
    raise RuntimeError('No hay transcripciones nuevas. Ejecuta la celda de transcripción.')
print('\nListo. Abre Mi unidad → MEIKA_Vox → Transcripciones.')